# 07 — Points de Gram et fonction de Riemann-Siegel

Dans le notebook 06, nos modèles ML butaient sur un problème fondamental : les gaps entre zéros sont très bruités.  
Ce notebook introduit un outil qui change la donne — la **fonction de Riemann-Siegel** et les **points de Gram**.

In [1]:
import numpy as np
import struct
import mpmath
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import Ridge
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

mpmath.mp.dps = 15

TEMPLATE     = "plotly_dark"
COLOR_MAIN   = "#7C6FCD"
COLOR_ACCENT = "#F2A623"
COLOR_MUTED  = "#888780"
COLOR_DANGER = "#E24B4A"
COLOR_TEAL   = "#1D9E75"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

Device : cpu


In [2]:
def parse_zeros_dat(filepath):
    with open(filepath, "rb") as f:
        raw = f.read()
    pos = 0
    B = struct.unpack_from('<Q', raw, pos)[0]; pos += 8
    all_zeros = []
    for block in range(B):
        t0  = struct.unpack_from('<d', raw, pos)[0]; pos += 8
        t1  = struct.unpack_from('<d', raw, pos)[0]; pos += 8
        Nt0 = struct.unpack_from('<Q', raw, pos)[0]; pos += 8
        Nt1 = struct.unpack_from('<Q', raw, pos)[0]; pos += 8
        n_zeros = Nt1 - Nt0
        cumsum = 0
        for _ in range(n_zeros):
            lo  = struct.unpack_from('<Q', raw, pos)[0]; pos += 8
            mid = struct.unpack_from('<I', raw, pos)[0]; pos += 4
            hi  = struct.unpack_from('<B', raw, pos)[0]; pos += 1
            z = (hi << 96) + (mid << 64) + lo
            cumsum += z
            all_zeros.append(t0 + cumsum * 2**-101)
    return np.array(all_zeros)

zeros_14 = parse_zeros_dat(r"C:\Users\Tangu\Zeta Riemann\data\zeros_14.dat")
gaps_14  = np.diff(zeros_14)
print(f"Zéros chargés : {len(zeros_14):,}")

Zéros chargés : 4,520


## 1. La fonction de Riemann-Siegel — d'où elle vient

Sur la ligne critique $s = \frac{1}{2} + it$, Riemann a montré qu'on peut factoriser ζ :

$$\zeta\left(\frac{1}{2} + it\right) = e^{-i\theta(t)} Z(t)$$

où :
- $\theta(t)$ est la **phase de Riemann-Siegel** — elle encode la rotation de ζ le long de la ligne critique
- $Z(t)$ est la **fonction de Riemann-Siegel** — elle est **réelle** pour t réel

$$\theta(t) = \text{Im}\left(\ln\Gamma\left(\frac{1}{4} + \frac{it}{2}\right)\right) - \frac{t}{2}\ln\pi$$

**Pourquoi c'est utile ?** On remplace un problème complexe par un problème réel.  
Les zéros de $\zeta(\frac{1}{2}+it)$ correspondent exactement aux zéros de $Z(t)$ — et $Z(t)$ est réelle, donc ses zéros sont des changements de signe, faciles à détecter.

**Pourquoi Γ apparaît ?** Elle est au cœur de l'équation fonctionnelle de ζ — elle encode la symétrie de ζ autour de la ligne critique. La phase θ(t) c'est exactement la rotation que subit ζ quand on se déplace sur cette ligne.

In [3]:
# Visualiser Z(t) et θ(t)
t_vals = np.linspace(10, 60, 3000)
Z_vals    = np.array([float(mpmath.siegelz(t))     for t in t_vals])
theta_vals = np.array([float(mpmath.siegeltheta(t)) for t in t_vals])

# Premiers zéros dans ce range
zeros_range = zeros_14[zeros_14 < 60]

fig = make_subplots(rows=2, cols=1,
    subplot_titles=("Z(t) — réelle, ses zéros = zéros non triviaux de ζ",
                    "θ(t) — phase croissante monotone"),
    vertical_spacing=0.15)

fig.add_trace(go.Scatter(
    x=t_vals, y=Z_vals, name="Z(t)",
    line=dict(color=COLOR_MAIN, width=1.5),
    fill="tozeroy", fillcolor="rgba(124,111,205,0.08)"
), row=1, col=1)
fig.add_hline(y=0, line_color=COLOR_MUTED, opacity=0.4, row=1, col=1)
fig.add_trace(go.Scatter(
    x=zeros_range, y=np.zeros(len(zeros_range)),
    mode="markers",
    marker=dict(color=COLOR_ACCENT, size=8),
    name="Zéros non triviaux"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=t_vals, y=theta_vals / np.pi,
    name="θ(t)/π",
    line=dict(color=COLOR_TEAL, width=1.5)
), row=2, col=1)

fig.update_layout(template=TEMPLATE, height=580,
    title="Fonction de Riemann-Siegel Z(t) et phase θ(t)")
fig.update_xaxes(title_text="t")
fig.update_yaxes(title_text="Z(t)", row=1, col=1)
fig.update_yaxes(title_text="θ(t)/π", row=2, col=1)
fig.show()

## 2. Les points de Gram — définition et calcul

Le n-ième **point de Gram** $g_n$ est défini par :

$$\theta(g_n) = n\pi$$

C'est la valeur de t où la phase θ a tourné de exactement n demi-tours.  
Entre $g_n$ et $g_{n+1}$, θ tourne de π — et Z(t) a "le temps" de faire en moyenne un zéro.

**La loi de Gram** : il y a en général exactement un zéro dans chaque intervalle $[g_n, g_{n+1}]$.  
Vérifiée empiriquement dans ~86% des cas sur nos données.

In [4]:
# Calculer les points de Gram dans le range de zeros_14.dat
theta_14   = float(mpmath.siegeltheta(14))
theta_5000 = float(mpmath.siegeltheta(5000))
n_start = int(np.ceil(theta_14   / np.pi))
n_end   = int(np.floor(theta_5000 / np.pi))

print(f"Calcul de {n_end - n_start + 1} points de Gram...")
gram_points = np.array([float(mpmath.grampoint(n)) for n in range(n_start, n_end+1)])
gram_gaps   = np.diff(gram_points)
print(f"Calculés : {len(gram_points)}")

print(f"\nEspacement moyen Gram  : {gram_gaps.mean():.6f}")
print(f"Espacement moyen zéros : {gaps_14.mean():.6f}")

# Loi de Gram : compter les zéros par intervalle
zeros_per_gram = np.array([
    np.sum((zeros_14 >= gram_points[i]) & (zeros_14 < gram_points[i+1]))
    for i in range(len(gram_points)-1)
])

total = len(zeros_per_gram)
print(f"\nLoi de Gram :")
for k in range(4):
    n = (zeros_per_gram == k).sum()
    print(f"  {k} zéros : {n:5d} ({n/total*100:.1f}%)")
print(f"\nLoi respectée (exactement 1 zéro) : {(zeros_per_gram==1).sum()/total*100:.1f}%")
print(f"Symétrie : {(zeros_per_gram==0).sum()} intervalles vides ↔ {(zeros_per_gram==2).sum()} intervalles doubles")

Calcul de 4520 points de Gram...
Calculés : 4520

Espacement moyen Gram  : 1.102422
Espacement moyen zéros : 1.103163

Loi de Gram :
  0 zéros :   320 (7.1%)
  1 zéros :  3882 (85.9%)
  2 zéros :   314 (6.9%)
  3 zéros :     3 (0.1%)

Loi respectée (exactement 1 zéro) : 85.9%
Symétrie : 320 intervalles vides ↔ 314 intervalles doubles


In [5]:
# Visualisation : zéros et points de Gram côte à côte
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=zeros_14[:50], y=np.zeros(50),
    mode="markers",
    marker=dict(color=COLOR_ACCENT, size=10, symbol="circle"),
    name="Zéros non triviaux"
))
fig.add_trace(go.Scatter(
    x=gram_points[:50], y=np.ones(50) * 0.3,
    mode="markers",
    marker=dict(color=COLOR_DANGER, size=8, symbol="diamond"),
    name="Points de Gram"
))
for g in gram_points[:25]:
    fig.add_vline(x=g, line_color=COLOR_DANGER, opacity=0.15, line_width=0.8)

fig.update_layout(
    template=TEMPLATE, height=320,
    title="Zéros non triviaux vs points de Gram — t ∈ [14, 120]",
    xaxis=dict(range=[14, 120]),
    yaxis=dict(visible=False)
)
fig.show()

## 3. Les déviations — une meilleure feature pour le ML

Au lieu de prédire le gap brut $\rho_{n+1} - \rho_n$, on prédit la **déviation** :

$$d_n = \rho_n - g_n$$

C'est la distance entre le n-ième zéro et le n-ième point de Gram.  
Cette quantité est plus petite et plus régulière — std réduite de 33% par rapport aux gaps bruts.

In [6]:
n_common   = min(len(zeros_14), len(gram_points))
deviations = zeros_14[:n_common] - gram_points[:n_common]

print("Déviations ρ_n − g_n :")
print(f"  mean   : {deviations.mean():.6f}")
print(f"  std    : {deviations.std():.6f}")
print(f"  min    : {deviations.min():.6f}")
print(f"  max    : {deviations.max():.6f}")
print(f"\nStd gaps bruts     : {gaps_14.std():.6f}")
print(f"Ratio std dev/gaps : {deviations.std()/gaps_14.std():.4f}  (−{(1-deviations.std()/gaps_14.std())*100:.1f}%)")

# Distribution des déviations
s_range = np.linspace(-4, 1, 300)
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=deviations, nbinsx=80,
    marker_color=COLOR_MAIN, opacity=0.85,
    name="ρ_n − g_n"
))
fig.add_vline(x=0, line_color=COLOR_ACCENT, line_dash="dot",
              annotation_text="g_n", annotation_font_color=COLOR_ACCENT)
fig.add_vline(x=deviations.mean(), line_color=COLOR_TEAL, line_dash="dot",
              annotation_text=f"mean={deviations.mean():.3f}",
              annotation_font_color=COLOR_TEAL)
fig.update_layout(
    template=TEMPLATE, height=380,
    title="Distribution des déviations ρ_n − g_n — forme GUE centrée en −0.55",
    xaxis_title="Déviation", yaxis_title="Fréquence"
)
fig.show()

Déviations ρ_n − g_n :
  mean   : -0.551681
  std    : 0.331042
  min    : -3.710874
  max    : 0.309556

Std gaps bruts     : 0.497747
Ratio std dev/gaps : 0.6651  (−33.5%)


## 4. ML sur les déviations — amélioration vs notebook 06

On reprend Ridge et LSTM du notebook 06 mais avec les déviations comme target.  
Résultats attendus : amélioration significative grâce à la réduction de variance.

In [7]:
k = 20
X_dev = np.array([deviations[i:i+k] for i in range(len(deviations)-k-1)])
y_dev = deviations[k+1:len(deviations)]

split = int(0.8 * len(X_dev))
X_tr, X_te = X_dev[:split], X_dev[split:]
y_tr, y_te = y_dev[:split], y_dev[split:]

# Baseline
baseline_mae_dev = np.mean(np.abs(y_te - deviations.mean()))

# Ridge
model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X_tr, y_tr)
pred_ridge = model_ridge.predict(X_te)
mae_ridge  = np.mean(np.abs(y_te - pred_ridge))

print(f"Baseline déviations : {baseline_mae_dev:.6f}")
print(f"Ridge déviations    : {mae_ridge:.6f}  (+{(baseline_mae_dev-mae_ridge)/baseline_mae_dev*100:.1f}%)")
print(f"\nPour rappel, Ridge sur gaps bruts (notebook 06) : +8.1%")

Baseline déviations : 0.234458
Ridge déviations    : 0.184801  (+21.2%)

Pour rappel, Ridge sur gaps bruts (notebook 06) : +8.1%


In [8]:
# LSTM sur les déviations
scaler     = StandardScaler()
dev_scaled = scaler.fit_transform(deviations.reshape(-1,1)).ravel()

X_sc = np.array([dev_scaled[i:i+k] for i in range(len(dev_scaled)-k-1)])
y_sc = dev_scaled[k+1:len(dev_scaled)]

split = int(0.8 * len(X_sc))
X_tr_sc = torch.FloatTensor(X_sc[:split].reshape(-1, k, 1))
y_tr_sc = torch.FloatTensor(y_sc[:split])
X_te_sc = torch.FloatTensor(X_sc[split:].reshape(-1, k, 1))
y_te_orig = y_dev[split:]

train_dl = DataLoader(TensorDataset(X_tr_sc, y_tr_sc), batch_size=256, shuffle=True)

class ZetaLSTM(nn.Module):
    def __init__(self, hidden=64, n_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, n_layers, batch_first=True, dropout=0.2)
        self.fc   = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

model_lstm = ZetaLSTM().to(device)
optim = torch.optim.Adam(model_lstm.parameters(), lr=1e-3)
crit  = nn.MSELoss()

for epoch in range(10):
    model_lstm.train()
    loss_total, n = 0, 0
    for X_b, y_b in train_dl:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optim.zero_grad()
        loss = crit(model_lstm(X_b), y_b)
        loss.backward()
        optim.step()
        loss_total += loss.item(); n += 1
    print(f"Epoch {epoch+1:2d}/10 — Loss : {loss_total/n:.6f}")

model_lstm.eval()
with torch.no_grad():
    pred_sc = model_lstm(X_te_sc.to(device)).cpu().numpy()
pred_lstm = scaler.inverse_transform(pred_sc.reshape(-1,1)).ravel()
mae_lstm  = np.mean(np.abs(y_te_orig - pred_lstm))

Epoch  1/10 — Loss : 0.970270
Epoch  2/10 — Loss : 0.940943
Epoch  3/10 — Loss : 0.831654
Epoch  4/10 — Loss : 0.712356
Epoch  5/10 — Loss : 0.688068
Epoch  6/10 — Loss : 0.657656
Epoch  7/10 — Loss : 0.667370
Epoch  8/10 — Loss : 0.589811
Epoch  9/10 — Loss : 0.593430
Epoch 10/10 — Loss : 0.581630


In [9]:
# Tableau comparatif complet
print("=" * 70)
print(f"{'Modèle':25s} {'MAE':>10} {'Amélioration':>15}")
print("=" * 70)
print(f"{'Baseline (gaps bruts)':25s} {'0.1678':>10} {'—':>15}")
print(f"{'Ridge (gaps bruts)':25s} {'0.1542':>10} {'+8.1%':>15}")
print(f"{'LSTM (gaps bruts)':25s} {'0.1852':>10} {'−10.4%':>15}")
print("-" * 70)
print(f"{'Baseline (déviations)':25s} {baseline_mae_dev:>10.4f} {'—':>15}")
print(f"{'Ridge (déviations)':25s} {mae_ridge:>10.4f} {f'+{(baseline_mae_dev-mae_ridge)/baseline_mae_dev*100:.1f}%':>15}")
print(f"{'LSTM (déviations)':25s} {mae_lstm:>10.4f} {f'+{(baseline_mae_dev-mae_lstm)/baseline_mae_dev*100:.1f}%':>15}")
print("=" * 70)

Modèle                           MAE    Amélioration
Baseline (gaps bruts)         0.1678               —
Ridge (gaps bruts)            0.1542           +8.1%
LSTM (gaps bruts)             0.1852          −10.4%
----------------------------------------------------------------------
Baseline (déviations)         0.2345               —
Ridge (déviations)            0.1848          +21.2%
LSTM (déviations)             0.1794          +23.5%


## Ce qu'on a découvert

| Concept | Résultat |
|---|---|
| **Fonction Z(t)** | Réelle — ses zéros = zéros non triviaux de ζ |
| **Phase θ(t)** | Croit monotonement — encode la rotation de ζ sur la ligne critique |
| **Points de Gram** | θ(g_n) = nπ — grille de référence régulière |
| **Loi de Gram** | 85.9% des intervalles contiennent exactement 1 zéro |
| **Symétrie** | Intervalles vides et doubles se compensent exactement |
| **Déviations** | Std réduite de 33% vs gaps bruts |
| **Ridge** | +21% vs +8% sur les gaps bruts — signal bien plus propre |
| **LSTM** | Comparable à Ridge — signal linéaire, pas besoin de non-linéarité |

### Ce qu'on pourrait faire pour aller plus loin

Les chercheurs ne s'arrêtent pas aux positions des zéros comme features.  
Ils utilisent les **valeurs de Z(t) aux points de Gram** — 21 points environnants + termes de la série de Riemann-Siegel.  
Ces 40 features encodent la structure locale de ζ autour de chaque intervalle, pas juste où se trouvent les zéros précédents.

C'est l'objet du notebook 08.

→ **Notebook 08 : Z(t) aux points de Gram comme features ML**